# EDA Municipal â€” AlfabetizaÃ§Ã£o no Brasil (Fase 3)

Base: `data/processed/features_municipio.parquet` + `targets_municipio.parquet` (5.500 municÃ­pios).
Gera figuras em `images/` e insumos para `reports/eda_municipio.md`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import DATA_PROCESSED, IMAGES_DIR
from src.visualization.plots import save_fig

feats = pd.read_parquet(DATA_PROCESSED / 'features_municipio.parquet')
targets = pd.read_parquet(DATA_PROCESSED / 'targets_municipio.parquet')
df = feats.merge(targets, on='co_municipio', validate='1:1')
df.shape

(5500, 62)

## 1. DistribuiÃ§Ã£o do resultado 2025

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['pc_alfabetizado_aeeb_2025'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('DistribuiÃ§Ã£o % alfabetizados (AEEB 2025)')
sns.boxplot(data=df, x='regiao', y='pc_alfabetizado_aeeb_2025', order=['N','NE','CO','SE','S'], ax=axes[1])
axes[1].set_title('Por regiÃ£o')
save_fig(fig, IMAGES_DIR / 'eda_01_distribuicao_resultado.png')
df.groupby('regiao')['pc_alfabetizado_aeeb_2025'].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
regiao,,,,,,,,
CO,467.0,81.2,12.8,33.0,72.7,83.3,91.2,100.0
N,450.0,62.5,15.1,12.8,51.7,61.8,73.8,96.9
NE,1790.0,69.6,17.8,9.1,56.4,69.9,83.7,100.0
S,1164.0,72.8,16.9,11.1,61.7,75.6,86.0,100.0
SE,1629.0,73.5,13.3,19.0,64.3,74.5,83.1,100.0


## 2. EvoluÃ§Ã£o 2023 â†’ 2024 â†’ 2025 por regiÃ£o

In [3]:
evo = df.groupby('regiao')[['pc_alfabetizado_2023','pc_alfabetizado_2024','pc_alfabetizado_aeeb_2025']].mean()
fig, ax = plt.subplots(figsize=(8, 4.5))
for reg in ['N','NE','CO','SE','S']:
    ax.plot([2023, 2024, 2025], evo.loc[reg], marker='o', label=reg)
ax.set_title('EvoluÃ§Ã£o da % de alfabetizados por regiÃ£o (mÃ©dia municipal)')
ax.set_ylabel('% alfabetizados'); ax.legend(title='RegiÃ£o')
save_fig(fig, IMAGES_DIR / 'eda_02_evolucao_regiao.png')
evo.round(1)

,pc_alfabetizado_2023,pc_alfabetizado_2024,pc_alfabetizado_aeeb_2025
regiao,,,
CO,64.6,72.3,81.2
N,45.4,49.1,62.5
NE,53.4,56.6,69.6
S,72.7,64.6,72.8
SE,62.2,70.1,73.5


## 3. Atingimento de meta 2025 por UF e por porte

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
uf_rate = df.groupby('sg_uf')['atingiu_meta_2025'].mean().sort_values()
uf_rate.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Taxa de atingimento da meta 2025 por UF')
axes[0].set_xlim(0, 1)
porte_rate = df.groupby('porte', observed=True)['atingiu_meta_2025'].mean()
porte_rate.plot.bar(ax=axes[1], color='darkorange')
axes[1].set_title('Por porte populacional'); axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=0)
save_fig(fig, IMAGES_DIR / 'eda_03_atingimento_meta.png')
print('Taxa geral de atingimento:', round(df['atingiu_meta_2025'].mean(), 3))

Taxa geral de atingimento: 0.725


## 4. CorrelaÃ§Ã£o (Spearman) com o resultado 2025

In [5]:
num = df.select_dtypes(include='number')
corr = num.corr(method='spearman')['pc_alfabetizado_aeeb_2025'].drop('pc_alfabetizado_aeeb_2025')
corr = corr.reindex(corr.abs().sort_values(ascending=False).index)
top = corr.head(18)
fig, ax = plt.subplots(figsize=(8, 7))
colors = ['seagreen' if v > 0 else 'indianred' for v in top.sort_values()]
top.sort_values().plot.barh(ax=ax, color=colors)
ax.set_title('CorrelaÃ§Ã£o de Spearman com % alfabetizados 2025 (top 18)')
save_fig(fig, IMAGES_DIR / 'eda_04_correlacao_spearman.png')
top.round(3)

pc_alfabetizado_2024      0.689
ambicao_2030             -0.689
gap_meta_2025             0.661
meta_2026                 0.574
meta_2027                 0.572
pc_alfabetizado_2023      0.571
meta_2028                 0.571
meta_2025                 0.571
meta_2024                 0.570
meta_2029                 0.568
esforco_2025             -0.452
gap_meta_2024             0.342
qtd_escolas              -0.326
qtd_escolas_municipais   -0.300
total_mat_fund_ai        -0.296
atingiu_meta_2024         0.293
total_tur_fund_ai        -0.290
total_mat_1_2_ano        -0.284
Name: pc_alfabetizado_aeeb_2025, dtype: float64

## 5. Scatters: IDHM-E, ruralidade, alunos/docente

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col, lab in zip(axes,
    ['idhm_e', 'pct_escolas_rurais', 'alunos_por_docente_ai'],
    ['IDHM EducaÃ§Ã£o (2010)', '% escolas rurais', 'Alunos por docente (AI)']):
    ax.scatter(df[col], df['pc_alfabetizado_aeeb_2025'], s=4, alpha=0.3)
    ax.set_xlabel(lab); ax.set_ylabel('% alfabetizados 2025')
    r = df[[col, 'pc_alfabetizado_aeeb_2025']].corr(method='spearman').iloc[0, 1]
    ax.set_title(f'Spearman rho = {r:.2f}')
save_fig(fig, IMAGES_DIR / 'eda_05_scatters_contexto.png')

## 6. Mapa coroplÃ©tico do gap 2025 (por UF)

In [7]:
import plotly.express as px

uf_gap = df.groupby('sg_uf')['gap_meta_2025'].mean().reset_index()
fig = px.choropleth(uf_gap, locations='sg_uf', color='gap_meta_2025',
                    scope='south america', color_continuous_scale='RdYlGn',
                    title='Gap mÃ©dio meta 2025 por UF (resultado - meta, p.p.)',
                    labels={'gap_meta_2025': 'gap (p.p.)', 'sg_uf': 'UF'})
fig.update_geos(visible=False, projection_type='mercator',
                center={'lat': -14, 'lon': -52}, lonaxis_range=[-75, -33], lataxis_range=[-35, 6])
fig.write_html(str(IMAGES_DIR / 'eda_06_mapa_gap_uf.html'))
try:
    fig.write_image(str(IMAGES_DIR / 'eda_06_mapa_gap_uf.png'), scale=2)
except Exception as e:
    print('PNG do mapa pulado (kaleido ausente):', e)
uf_gap.sort_values('gap_meta_2025').head()

PNG do mapa pulado (kaleido ausente): 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"



,sg_uf,gap_meta_2025
21,RS,-8.876923
2,AM,0.216667
17,RJ,0.945055
24,SP,1.184000
22,SC,1.798587


## 7. Matriz de valores faltantes

In [8]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(miss))))
(miss * 100).plot.barh(ax=ax, color='slategray')
ax.set_title('% de valores faltantes por coluna')
save_fig(fig, IMAGES_DIR / 'eda_07_missing.png')
(miss * 100).round(2)

va_agropecuaria               100.00
share_va_agro                 100.00
va_servicos                   100.00
va_industria                  100.00
va                            100.00
delta_2023_2024                 5.71
atingiu_meta_2024               3.62
gap_meta_2024                   3.62
pc_alfabetizado_2024            3.62
esforco_2025                    3.62
ambicao_2030                    3.62
pc_alfabetizado_2023            3.60
meta_2025                       1.51
meta_2024                       1.51
atingiu_meta_2025               1.51
gap_meta_2025                   1.51
meta_2030                       0.67
meta_2026                       0.67
meta_2028                       0.67
meta_2029                       0.67
meta_2027                       0.67
idhm_e                          0.11
idhm_r                          0.11
idhm_l                          0.11
idhm                            0.11
taxa_analfabetismo_15_mais      0.11
expectativa_anos_estudo         0.11
i

## 8. EsforÃ§o exigido pela meta Ã— atingimento (H4)

In [9]:
fig, ax = plt.subplots(figsize=(7, 4.5))
d8 = df.dropna(subset=['atingiu_meta_2025']).copy()
d8['atingiu_meta_2025'] = d8['atingiu_meta_2025'].astype(int)
sns.boxplot(data=d8, x='atingiu_meta_2025', y='esforco_2025', ax=ax, showfliers=False)
ax.set_xticklabels(['NÃ£o atingiu', 'Atingiu'])
ax.set_title('EsforÃ§o exigido (meta_2025 - resultado_2024) Ã— atingimento')
ax.set_ylabel('esforÃ§o (p.p.)')
save_fig(fig, IMAGES_DIR / 'eda_08_esforco_vs_atingimento.png')
d8.groupby('atingiu_meta_2025')['esforco_2025'].describe().round(2)


C:\Users\icaro\AppData\Local\Temp\ipykernel_3200\2247050602.py:5: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(['NÃ£o atingiu', 'Atingiu'])


,count,mean,std,min,25%,50%,75%,max
atingiu_meta_2025,,,,,,,,
0,1422.0,11.34,13.59,-47.0,3.0,11.0,19.0,69.0
1,3879.0,-1.21,13.73,-76.0,-10.0,-1.0,7.0,55.0


## 9. HipÃ³teses formalizadas

Consolidadas em `reports/eda_municipio.md` (H1â€“H5), ligadas Ã s decisÃµes de modelagem.